# SHL Grammar Scoring Engine
Research Engineer assessment - scoring spoken grammar 0-5 from 45-60s audio.

My plan: transcribe with Whisper (grammar is in the words), then TF-IDF + simple linguistic stats + a few acoustic features, Ridge + LightGBM ensemble. Keeping it simple so I can explain it in the interview.

In [ ]:
# run this first on Kaggle
import sys
try:
    import whisper
    print('whisper ok')
except ImportError:
    print('installing whisper + lightgbm...')
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','-q','openai-whisper','lightgbm','soundfile'])
    import whisper
    print('installed')


In [ ]:
import os, re, gc, warnings, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8,4)

# paths - works both on kaggle and locally
if os.path.exists('/kaggle/input'):
    # adjust this to your kaggle dataset path
    import glob
    cands = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    print(cands)
    ROOT = Path(cands[0]).parent if cands else (Path('Dataset_Final') if Path('Dataset_Final').exists() else Path('/kaggle/input/shl-hiring-assessment-2026/Dataset_Final') if Path('/kaggle/input/shl-hiring-assessment-2026/Dataset_Final').exists() else Path('.'))
else:
    ROOT = Path('Dataset_Final')
print('ROOT:', ROOT)
TRAIN_CSV = ROOT/'train.csv'
TEST_CSV = ROOT/'test.csv'
TRAIN_DIR = ROOT/'train'
TEST_DIR = ROOT/'test'


In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
print('train', train.shape, 'test', test.shape)
print(train.head())
print(train['label'].describe())
print(test.head())

In [ ]:
# label distribution - looks clustered at 3.0 and 5.0
plt.figure()
plt.hist(train['label'], bins=20, edgecolor='black')
plt.xlabel('grammar score')
plt.ylabel('count')
plt.title('Train label distribution (n=769)')
plt.show()
print(train['label'].value_counts().sort_index())

# quick check: same filename != same audio (they reused names)
import hashlib
def md5(p):
    return hashlib.md5(open(p,'rb').read()).hexdigest()[:8]
for fn in ['audio_19.wav','audio_0.wav']:
    a = TRAIN_DIR/fn; b = TEST_DIR/fn
    if a.exists() and b.exists():
        print(fn, md5(a), md5(b), 'same content?' , md5(a)==md5(b))

In [ ]:
import soundfile as sf
# duration stats (fast, no librosa needed here)
def get_durs(folder, csv, label_col=None):
    df = pd.read_csv(csv)
    durs = []
    for fn in tqdm(df['filename'], desc=f'durations {folder.name}'):
        try:
            durs.append(sf.info(str(folder/fn)).duration)
        except:
            durs.append(np.nan)
    df['duration'] = durs
    return df

train = get_durs(TRAIN_DIR, TRAIN_CSV)
# reload label since get_durs re-read csv without it? merge back
_lab = pd.read_csv(TRAIN_CSV)[['filename','label']]
# actually train.csv already had label; keep it
train['label'] = _lab['label']
test2 = get_durs(TEST_DIR, TEST_CSV)
print(train[['label','duration']].head())
print('corr duration vs label:', train['label'].corr(train['duration']))

plt.figure()
plt.scatter(train['duration'], train['label'], alpha=0.3)
plt.xlabel('duration (s)')
plt.ylabel('score')
plt.title('Duration vs grammar score (weak)')
plt.show()

plt.figure()
# avg duration per score bucket
print(train.groupby('label')['duration'].mean())

## 1. Preprocessing
Audio is already 16kHz mono. I just trim leading/trailing silence and peak-normalize. No denoising - wanted to keep it simple and not break whisper.

In [ ]:
import librosa
SR = 16000
def load_audio(path, sr=SR):
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    # trim silence at ends
    y, _ = librosa.effects.trim(y, top_db=30)
    # peak norm, avoid div by 0
    m = np.max(np.abs(y))
    if m > 0:
        y = y / m * 0.9
    return y.astype(np.float32)

# sanity check one file
p = list(TRAIN_DIR.glob('*.wav'))[0]
print(p)
y = load_audio(p)
print('loaded', y.shape, 'mean', y.mean(), 'max', abs(y).max())
plt.plot(y[::200][:2000])
plt.title('waveform snippet (downsampled)')
plt.show()

## 2. Transcription with Whisper
Grammar lives in the words, so ASR quality matters most. Using whisper `base` (good speed/accuracy tradeoff on P100). Results cached to csv so re-runs are quick.

In [ ]:
import whisper
# change to 'small' if you have time/GPU headroom, 'tiny' if in a hurry
MODEL_NAME = 'small'  # v2: better ASR for grammar
print('loading whisper', MODEL_NAME)
model = whisper.load_model(MODEL_NAME)

def transcribe_folder(file_list, folder, out_col='transcript'):
    rows = []
    for fn in tqdm(file_list, desc=f'transcribe {folder.name}'):
        p = folder/fn
        try:
            # whisper handles loading + resampling itself
            res = model.transcribe(str(p), fp16=False, language='en')
            txt = res.get('text','').strip()
        except Exception as e:
            print('fail', fn, e)
            txt = ''
        rows.append((fn, txt))
    return pd.DataFrame(rows, columns=['filename', out_col])

# resume from cache if exists
CACHE_TR = Path('transcripts_train.csv')
CACHE_TE = Path('transcripts_test.csv')
if CACHE_TR.exists() and CACHE_TE.exists():
    tr_txt = pd.read_csv(CACHE_TR)
    te_txt = pd.read_csv(CACHE_TE)
    print('loaded cache', tr_txt.shape, te_txt.shape)
else:
    tr_txt = transcribe_folder(train['filename'].tolist(), TRAIN_DIR)
    tr_txt.to_csv(CACHE_TR, index=False)
    te_txt = transcribe_folder(test['filename'].tolist(), TEST_DIR)
    te_txt.to_csv(CACHE_TE, index=False)
    print('saved')
print(tr_txt.head())
print(te_txt.head())
# free some gpu mem
# import torch; del model; torch.cuda.empty_cache()

In [ ]:
# look at a few examples by score
tmp = pd.read_csv(TRAIN_CSV)[['filename','label']].merge(tr_txt, on='filename')
for s in [5.0, 2.0, 0.0]:
    print('--- score', s, '---')
    rows = tmp[tmp['label']==s].head(2)
    for _, r in rows.iterrows():
        print(r['filename'], ':', r['transcript'][:300])
    print()

# transcript length vs score - should correlate
tmp['n_words'] = tmp['transcript'].fillna('').apply(lambda x: len(x.split()))
print('corr n_words vs label:', tmp['n_words'].corr(tmp['label']))
plt.scatter(tmp['n_words'], tmp['label'], alpha=0.3)
plt.xlabel('n_words in transcript')
plt.ylabel('score')
plt.title('Transcript length vs score')
plt.show()

## 3. Features
Two groups:
- linguistic: from whisper transcripts (word count, wpm, lexical diversity, avg word len, sentence stats + TF-IDF)
- acoustic: from waveform (mfcc means, zcr, rms, centroid, silence-ish stats)
Nothing fancy, all explainable.

In [ ]:
STOPWORDS = set('''a an the and or but if then else when at by for with about into through during before after above below to from up down in out on off over under again further once here there all any both each few more most other some such no nor not only own same so than too very can will just don should now i you he she it we they me him her us them my your his their our'''.split())

def ling_features(df_txt, durations):
    # df_txt: filename, transcript ; durations: dict fn->dur
    feats = []
    for _, row in df_txt.iterrows():
        t = str(row['transcript'] if pd.notna(row['transcript']) else '')
        words = re.findall(r"[a-z']+", t.lower())
        nw = len(words)
        nchar = len(t)
        nsent = max(1, t.count('.') + t.count('?') + t.count('!') + 1)
        uniq = len(set(words)) if words else 0
        ttr = uniq / nw if nw else 0
        avg_wlen = np.mean([len(w) for w in words]) if words else 0
        dur = durations.get(row['filename'], 55.0)
        wpm = nw / (dur/60.0) if dur else 0
        stop_r = sum(1 for w in words if w in STOPWORDS)/nw if nw else 0
        long_r = sum(1 for w in words if len(w)>6)/nw if nw else 0
        feats.append([nw, nchar, nsent, nw/max(1,nsent), uniq, ttr, avg_wlen, wpm, stop_r, long_r, dur])
    cols = ['n_words','n_chars','n_sent','w_per_sent','n_uniq','ttr','avg_wlen','wpm','stop_ratio','long_ratio','duration']
    out = pd.DataFrame(feats, columns=cols)
    out['filename'] = df_txt['filename'].values
    return out

dur_train = dict(zip(train['filename'], train['duration']))
# test durations
import soundfile as asf
te_durs = {}
for fn in test['filename']:
    try:
        te_durs[fn] = asf.info(str(TEST_DIR/fn)).duration
    except:
        te_durs[fn] = 55.0

Xling_tr = ling_features(tr_txt, dur_train)
Xling_te = ling_features(te_txt, te_durs)
print(Xling_tr.describe().T)
print(Xling_tr.head())

In [ ]:
def acoustic_features(fn_list, folder):
    rows = []
    for fn in tqdm(fn_list, desc=f'acoustic {folder.name}'):
        try:
            y = load_audio(folder/fn)
            mf = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=13)
            mf_mean = mf.mean(axis=1)
            mf_std = mf.std(axis=1)
            zcr = float(librosa.feature.zero_crossing_rate(y).mean())
            rms = float(np.sqrt((y**2).mean()))
            cent = float(librosa.feature.spectral_centroid(y=y, sr=SR).mean())
            # silence ratio: frac of frames with low energy
            frame_e = librosa.feature.rms(y=y)[0]
            sil = float((frame_e < 0.02).mean())
            rows.append([zcr, rms, cent, sil] + mf_mean.tolist() + mf_std.tolist())
        except Exception as e:
            print('acoustic fail', fn, e)
            rows.append([0]*30)
    cols = ['zcr','rms','cent','sil_ratio'] + [f'mfcc{i}_m' for i in range(13)] + [f'mfcc{i}_s' for i in range(13)]
    out = pd.DataFrame(rows, columns=cols)
    out['filename'] = fn_list
    return out

Xac_tr = acoustic_features(train['filename'].tolist(), TRAIN_DIR)
Xac_te = acoustic_features(test['filename'].tolist(), TEST_DIR)
print(Xac_tr.head())

## 4. Model - Ridge + LightGBM ensemble
Why these: Ridge on TF-IDF works well for text regression and is easy to explain; LightGBM picks up non-linear bits from numeric features. Average them. 5-fold CV, metrics RMSE + Pearson (kaggle eval).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import lightgbm as lgb

# TF-IDF on transcripts
all_txt = pd.concat([tr_txt['transcript'].fillna(''), te_txt['transcript'].fillna('')])
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english', sublinear_tf=True)
vec.fit(all_txt)
print('vocab', len(vec.vocabulary_))
Ttr = vec.transform(tr_txt['transcript'].fillna(''))
Tte = vec.transform(te_txt['transcript'].fillna(''))

# numeric matrix
num_cols = [c for c in Xling_tr.columns if c!='filename']
ac_cols = [c for c in Xac_tr.columns if c!='filename']
# merge ling+acoustic on filename order
order_tr = train['filename'].tolist()
order_te = test['filename'].tolist()
Ntr = Xling_tr.set_index('filename').loc[order_tr].reset_index(drop=True)
Atr = Xac_tr.set_index('filename').loc[order_tr].reset_index(drop=True)
Nte = Xling_te.set_index('filename').loc[order_te].reset_index(drop=True)
Ate = Xac_te.set_index('filename').loc[order_te].reset_index(drop=True)

from scipy.sparse import hstack, csr_matrix
Xtr_num = pd.concat([Ntr, Atr[ac_cols]], axis=1).values
Xte_num = pd.concat([Nte, Ate[ac_cols]], axis=1).values
print('numeric shapes', Xtr_num.shape, Xte_num.shape, 'tfidf', Ttr.shape)

y = pd.read_csv(TRAIN_CSV)['label'].values  # keep original order? ensure aligned
# align y to order_tr
ymap = dict(zip(pd.read_csv(TRAIN_CSV)['filename'], pd.read_csv(TRAIN_CSV)['label']))
y = np.array([ymap[f] for f in order_tr], dtype=float)
print('y mean', y.mean(), 'std', y.std())

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_ridge = np.zeros(len(y))
oof_lgb = np.zeros(len(y))

import numpy as np
for fold, (tr_i, va_i) in enumerate(kf.split(Xtr_num)):
    # --- ridge on tfidf ---
    rr = RidgeCV(alphas=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0])
    rr.fit(Ttr[tr_i], y[tr_i])
    oof_ridge[va_i] = rr.predict(Ttr[va_i])
    # --- lgbm on numeric ---
    dtr = lgb.Dataset(Xtr_num[tr_i], label=y[tr_i])
    dva = lgb.Dataset(Xtr_num[va_i], label=y[va_i])
    params = {'objective':'regression','metric':'rmse','verbosity':-1,'boosting_type':'gbdt',
              'learning_rate':0.05,'num_leaves':31,'feature_fraction':0.8,'bagging_fraction':0.8,'bagging_freq':1,'seed':fold}
    m = lgb.train(params, dtr, num_boost_round=500, valid_sets=[dva], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[va_i] = m.predict(Xtr_num[va_i], num_iteration=m.best_iteration)
    rm = mean_squared_error(y[va_i], (oof_ridge[va_i]+oof_lgb[va_i])/2)**0.5
    print(f'fold {fold} rmse-avg {rm:.4f}')

oof_avg = (oof_ridge + oof_lgb)/2
oof_avg = np.clip(oof_avg, 0, 5)
rmse = mean_squared_error(y, oof_avg)**0.5
pear, _ = pearsonr(y, oof_avg)
print(f'CV RMSE: {rmse:.4f}  Pearson: {pear:.4f}')
print(f"Ridge only RMSE {mean_squared_error(y, np.clip(oof_ridge,0,5))**0.5:.4f} pear {pearsonr(y, oof_ridge)[0]:.4f}")
print(f"LGBM only RMSE {mean_squared_error(y, np.clip(oof_lgb,0,5))**0.5:.4f} pear {pearsonr(y, oof_lgb)[0]:.4f}")

In [ ]:
# plots
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].scatter(y, oof_avg, alpha=0.4)
ax[0].plot([0,5],[0,5],'r--')
ax[0].set_xlabel('true')
ax[0].set_ylabel('pred (oob)')
ax[0].set_title(f'Pred vs True (pear={pear:.3f})')
ax[1].hist(y-oof_avg, bins=30, edgecolor='black')
ax[1].set_xlabel('residual (true-pred)')
ax[1].set_title(f'Residuals CV RMSE={rmse:.3f}')
plt.show()

# quick feature importance from last lgb model (retrain quick on full for plot)
# train one lgb on full numeric for importance viz
dall = lgb.Dataset(Xtr_num, label=y)
pall = {'objective':'regression','verbosity':-1,'learning_rate':0.05,'num_leaves':31,'seed':0}
ball = lgb.train(pall, dall, num_boost_round=200)
imp = pd.Series(ball.feature_importance(importance_type='gain'), index=list(num_cols)+ac_cols).sort_values(ascending=False).head(20)
plt.figure()
imp[::-1].plot(kind='barh')
plt.title('Top numeric features (gain)')
plt.tight_layout()
plt.show()

## 4b. v2 upgrades for RMSE
Switched whisper to `small`, added MiniLM embeddings + DistilBERT fine-tune, tuned weighted ensemble + 0.5 rounding. Still the same simple story: better transcripts -> better text regression.

v3 added: CoLA acceptability + disfluency heuristics (grammar Ridge). Still explainable: acceptable-sentence rate tracks rubric directly.


In [ ]:
# v2: sentence embeddings (MiniLM) + Ridge - cheap and strong for grammar
try:
    from sentence_transformers import SentenceTransformer
    USE_SBERT = True
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
    from sentence_transformers import SentenceTransformer
    USE_SBERT = True

print('encoding transcripts with MiniLM...')
_sbert = SentenceTransformer('all-MiniLM-L6-v2')
Etr = _sbert.encode(tr_txt['transcript'].fillna('').tolist(), show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
Ete = _sbert.encode(te_txt['transcript'].fillna('').tolist(), show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
print('emb shapes', Etr.shape, Ete.shape)

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import numpy as np

# OOF for sbert-ridge (same folds as before for fair averaging)
kf2 = KFold(n_splits=5, shuffle=True, random_state=42)
oof_sbert = np.zeros(len(y))
for tr_i, va_i in kf2.split(Etr):
    rr = RidgeCV(alphas=[0.1, 0.5, 1.0, 5.0, 20.0])
    rr.fit(Etr[tr_i], y[tr_i])
    oof_sbert[va_i] = rr.predict(Etr[va_i])
print('SBERT Ridge OOF RMSE', mean_squared_error(y, np.clip(oof_sbert,0,5))**0.5, 'pear', pearsonr(y, oof_sbert)[0])


In [ ]:
# v2: fine-tune DistilBERT on transcripts (optional, big boost if GPU allows)
# set USE_BERT=False to skip and still submit with ridge+lgbm+sbert
USE_BERT = True
oof_bert = np.zeros(len(y))
pred_bert_test_folds = []

if USE_BERT:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
        from datasets import Dataset
        print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers','datasets','accelerate'])
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
        from datasets import Dataset

    MODEL_ID = 'distilbert-base-uncased'
    tok = AutoTokenizer.from_pretrained(MODEL_ID)

    def tokenize_tr_te(tr_txt_list, te_txt_list, max_len=384):
        # transcripts are ~150-250 words, 384 covers most, faster than 512
        t_tr = tok(tr_txt_list, truncation=True, padding='max_length', max_length=max_len)
        t_te = tok(te_txt_list, truncation=True, padding='max_length', max_length=max_len)
        return t_tr, t_te

    tr_list = tr_txt['transcript'].fillna('').tolist()
    te_list = te_txt['transcript'].fillna('').tolist()
    # need order aligned to y (order_tr)
    # tr_txt order may differ from order_tr, so reorder
    txt_map = dict(zip(tr_txt['filename'], tr_txt['transcript'].fillna('')))
    tr_list_ordered = [str(txt_map.get(f,'')) for f in order_tr]
    # test already in order_te
    te_map = dict(zip(te_txt['filename'], te_txt['transcript'].fillna('')))
    te_list_ordered = [str(te_map.get(f,'')) for f in order_te]

    t_tr, t_te = tokenize_tr_te(tr_list_ordered, te_list_ordered)

    import numpy as np
    kfB = KFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (tri, vai) in enumerate(kfB.split(np.arange(len(y)))):
        print(f'--- BERT fold {fold} ---')
        def make_ds(ids, labels=None):
            d = {'input_ids': [t_tr['input_ids'][i] for i in ids],
                 'attention_mask': [t_tr['attention_mask'][i] for i in ids]}
            if labels is not None:
                d['labels'] = [float(labels[i]) for i in ids]
            return Dataset.from_dict(d)
        ds_tr = make_ds(tri, y)
        ds_va = make_ds(vai, y)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=1, problem_type='regression')
        args = TrainingArguments(
            output_dir=f'./bert_fold{fold}', num_train_epochs=3,
            per_device_train_batch_size=8, per_device_eval_batch_size=16,
            learning_rate=2e-5, weight_decay=0.01,
            eval_strategy='epoch', save_strategy='no',
            logging_strategy='no', seed=42+fold, fp16=torch.cuda.is_available(),
            load_best_model_at_end=False)
        def comp(p):
            pred = p.predictions.flatten()
            return {'rmse': float(mean_squared_error(p.label_ids, pred)**0.5)}
        trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va, compute_metrics=comp)
        trainer.train()
        out = trainer.predict(ds_va)
        oof_bert[vai] = out.predictions.flatten()
        # test preds for this fold
        ds_te = Dataset.from_dict({'input_ids': t_te['input_ids'], 'attention_mask': t_te['attention_mask']})
        tp = trainer.predict(ds_te).predictions.flatten()
        pred_bert_test_folds.append(tp)
        del model, trainer
        import gc; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'fold {fold} OOF RMSE so far:', mean_squared_error(y[np.concatenate([np.arange(len(y))[ :1]])], [0]) if False else '')
    print('BERT OOF RMSE', mean_squared_error(y, np.clip(oof_bert,0,5))**0.5, 'pear', pearsonr(y, oof_bert)[0])
else:
    print('skip BERT')


In [ ]:
# v3: grammar-specific signal (fast, no long training)
# CoLA acceptability + disfluency heuristics - directly targets 1-5 rubric
import re, numpy as np, torch
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# --- 1) disfluency / heuristic grammar proxies (offline, instant) ---
FILLERS = set(['um','uh','hmm','like','actually','basically','literally','stuff','things','yeah','okay'])
def heuristic_feats(txt):
    t = str(txt if isinstance(txt, str) else '')
    words = re.findall(r"[a-z']+", t.lower())
    nw = len(words)
    sents = [s.strip() for s in re.split(r'[.!?]+', t) if s.strip()]
    ns = max(1, len(sents))
    fill = sum(1 for w in words if w in FILLERS)/nw if nw else 0
    # repetition: frac of repeated adjacent bigrams
    bigrams = list(zip(words, words[1:])) if nw>1 else []
    rep = 0
    if bigrams:
        from collections import Counter
        c = Counter(bigrams)
        rep = sum(1 for v in c.values() if v>1)/max(1,len(c))
    avg_sent_len = nw/ns
    short_sent_rate = sum(1 for s in sents if len(s.split())<4)/ns
    return [fill, rep, avg_sent_len, short_sent_rate, ns, nw]

tr_list_h = tr_txt['transcript'].fillna('').tolist()
te_list_h = te_txt['transcript'].fillna('').tolist()
Htr = np.array([heuristic_feats(t) for t in tr_list_h])
Hte = np.array([heuristic_feats(t) for t in te_list_h])
print('heur shapes', Htr.shape, Hte.shape)

# --- 2) CoLA acceptability per sentence (pretrained, GPU, few minutes) ---
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    print('loading CoLA model...')
    cola_id = 'textattack/roberta-base-CoLA'
    ctok = AutoTokenizer.from_pretrained(cola_id)
    cmod = AutoModelForSequenceClassification.from_pretrained(cola_id)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cmod.to(dev).eval()
    print('cola on', dev)

    @torch.no_grad()
    def cola_sent_scores(texts, batch=32, max_len=128):
        # split each transcript into sentences, score each, aggregate
        import torch.nn.functional as F
        feats = []
        for t in texts:
            sents = [s.strip() for s in re.split(r'[.!?]+', str(t)) if s.strip() and len(s.split())>=2]
            if not sents:
                feats.append([0.5, 0.0, 0.5, 0.0])
                continue
            probs = []
            for i in range(0, len(sents), batch):
                chunk = sents[i:i+batch]
                enc = ctok(chunk, truncation=True, padding=True, max_length=max_len, return_tensors='pt').to(dev)
                logits = cmod(**enc).logits
                pr = torch.softmax(logits, dim=-1)[:,1].cpu().numpy()  # P(acceptable)
                probs.extend(pr.tolist())
            probs = np.array(probs)
            feats.append([float(probs.mean()), float(probs.std()), float((probs>0.5).mean()), float(probs.min())])
        return np.array(feats)

    Ctr = cola_sent_scores(tr_txt['transcript'].fillna('').tolist())
    Cte = cola_sent_scores(te_list_h)
    print('cola shapes', Ctr.shape, Cte.shape)
    print('cola mean vs label corr:', pearsonr(Ctr[:,0], y)[0])
    Gtr = np.hstack([Htr, Ctr])
    Gte = np.hstack([Hte, Cte])
except Exception as e:
    print('CoLA failed, heuristics only:', e)
    Gtr, Gte = Htr, Hte

# --- 3) OOF Ridge on grammar feats (same folds) ---
kfG = KFold(n_splits=5, shuffle=True, random_state=42)
oof_grammar = np.zeros(len(y))
for tri, vai in kfG.split(Gtr):
    rr = RidgeCV(alphas=[0.1, 0.5, 1.0, 5.0, 20.0])
    rr.fit(Gtr[tri], y[tri])
    oof_grammar[vai] = rr.predict(Gtr[vai])
print('GRAMMAR Ridge OOF RMSE', mean_squared_error(y, np.clip(oof_grammar,0,5))**0.5, 'pear', pearsonr(y, oof_grammar)[0])

# refit on full for test
gram_final = RidgeCV(alphas=[0.1,0.5,1.0,5.0,20.0])
gram_final.fit(Gtr, y)
pred_grammar_te = np.clip(gram_final.predict(Gte), 0, 5)
print('grammar test preds', pred_grammar_te[:5])


In [ ]:
# v2: weighted ensemble tuned for RMSE (uses existing oof_ridge, oof_lgb + new oof_sbert, oof_bert)
import itertools
cands = {
    'ridge': np.clip(oof_ridge,0,5),
    'lgb': np.clip(oof_lgb,0,5),
    'sbert': np.clip(oof_sbert,0,5),
}
if 'oof_bert' in globals() and np.abs(oof_bert).sum() > 0:
    cands['bert'] = np.clip(oof_bert,0,5)
if 'oof_grammar' in globals():
    cands['grammar'] = np.clip(oof_grammar,0,5)

names = list(cands.keys())
M = np.stack([cands[k] for k in names], axis=1)
print('models:', names)
# quick grid search on simplex (step 0.2) for lowest RMSE
best = None
best_w = None
for w in itertools.product([0,0.2,0.4,0.6,0.8,1.0], repeat=len(names)):
    if abs(sum(w)-1.0) > 1e-6 or sum(1 for x in w if x>0)==0:
        continue
    p = (M*np.array(w)).sum(axis=1)
    p = np.round(np.clip(p,0,5)*2)/2  # match submission rounding
    rm = mean_squared_error(y, p)**0.5
    if best is None or rm < best:
        best, best_w = rm, w
print('best OOF RMSE', round(float(best),4), 'weights', dict(zip(names, best_w)))

# refit sbert ridge on full data for test
from sklearn.linear_model import RidgeCV
sbert_final = RidgeCV(alphas=[0.1,0.5,1.0,5.0,20.0])
sbert_final.fit(Etr, y)
pred_sbert_te = np.clip(sbert_final.predict(Ete),0,5)

# combine test preds with same weights
# grammar test already in pred_grammar_te
# final_ridge/final_lgb already fit on full data in next cell - so save weights and apply after
ENSEMBLE_WEIGHTS = dict(zip(names, best_w))
print('save weights for next cell:', ENSEMBLE_WEIGHTS)
# bert test mean
if 'pred_bert_test_folds' in globals() and len(pred_bert_test_folds)>0:
    pred_bert_te = np.mean(pred_bert_test_folds, axis=0)
    print('bert test mean', pred_bert_te[:3])


## 4d. v5 LGBM-all (fast new preds)
Old LGBM never saw CoLA/grammar. This one does, so preds actually change.


In [ ]:
# v5 FAST new signal: LGBM now sees grammar (CoLA+heur) too (5-10 min, no re-transcribe)
# old lgb only saw ling+acoustic; this one sees all numeric -> different preds
import numpy as np, lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from scipy.optimize import nnls

Xall = np.hstack([Xtr_num, Gtr])
Xall_te = np.hstack([Xte_num, Gte])
print('Xall', Xall.shape, Xall_te.shape)

kfA = KFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb_all = np.zeros(len(y))
pred_lgb_all_folds = []
for fold,(tri,vai) in enumerate(kfA.split(Xall)):
    dtr=lgb.Dataset(Xall[tri], label=y[tri])
    dva=lgb.Dataset(Xall[vai], label=y[vai])
    par={'objective':'regression','metric':'rmse','verbosity':-1,'learning_rate':0.03,'num_leaves':31,'feature_fraction':0.8,'bagging_fraction':0.8,'bagging_freq':1,'seed':fold}
    m=lgb.train(par,dtr,num_boost_round=800,valid_sets=[dva],callbacks=[lgb.early_stopping(80,verbose=False)])
    oof_lgb_all[vai]=m.predict(Xall[vai],num_iteration=m.best_iteration)
    pred_lgb_all_folds.append(m.predict(Xall_te,num_iteration=m.best_iteration))
    print(f'fold {fold} best_iter {m.best_iteration} rmse {mean_squared_error(y[vai],oof_lgb_all[vai])**0.5:.4f}')
print('LGB-ALL OOF RMSE',mean_squared_error(y,np.clip(oof_lgb_all,0,5))**0.5,'pear',pearsonr(y,oof_lgb_all)[0])

# restack everything incl new model
_c={}; _c['ridge']=np.clip(oof_ridge,0,5); _c['lgb']=np.clip(oof_lgb,0,5); _c['lgb_all']=np.clip(oof_lgb_all,0,5)
if 'oof_sbert' in globals(): _c['sbert']=np.clip(oof_sbert,0,5)
if 'oof_grammar' in globals(): _c['grammar']=np.clip(oof_grammar,0,5)
_ns=list(_c.keys()); _M=np.stack([_c[k] for k in _ns],axis=1)
_w,_=nnls(_M,y); _w=_w/_w.sum()
print('NNLS2',dict(zip(_ns,np.round(_w,3))))
_p=_M@_w; print('NNLS2 OOF RMSE',mean_squared_error(y,np.round(np.clip(_p,0,5)*2)/2)**0.5)

# test side
pred_lgb_all_te=np.mean(pred_lgb_all_folds,axis=0)
_parts={'ridge':final_ridge.predict(Tte),'lgb':final_lgb.predict(Xte_num),'lgb_all':pred_lgb_all_te}
if 'pred_sbert_te' in globals(): _parts['sbert']=pred_sbert_te
if 'pred_grammar_te' in globals(): _parts['grammar']=pred_grammar_te
_wmap=dict(zip(_ns,_w))
_order=[k for k in _wmap if k in _parts]; _s=sum(_wmap[k] for k in _order)
pred_v5=np.zeros_like(_parts[_order[0]],dtype=float)
for k in _order: pred_v5+=_parts[k]*_wmap[k]/_s
if globals().get('USE_ISO',False):
    try: pred_v5=np.clip(iso.predict(np.clip(pred_v5,0,5)),0,5)
    except: pass
pred_v5=np.round(np.clip(pred_v5,0,5)*2)/2
import pandas as pd
sub5=pd.DataFrame({'filename':order_te,'label':pred_v5})
sub5.to_csv('submission_v5.csv',index=False)
print('saved submission_v5.csv',sub5.shape)
print(sub5.head())
# diff check vs old submission
try:
    old=pd.read_csv('submission.csv')
    print('mean abs change vs submission.csv:',(abs(old['label'].values-pred_v5)).mean())
except Exception as e: print(e)


## 5. Final fit on full train + train RMSE (required)
Refit both models on all data, report train RMSE here as required by the brief.

In [ ]:
# final ridge
final_ridge = RidgeCV(alphas=[0.1,0.5,1.0,2.0,5.0,10.0])
final_ridge.fit(Ttr, y)
# final lgb
dall = lgb.Dataset(Xtr_num, label=y)
pall = {'objective':'regression','metric':'rmse','verbosity':-1,'learning_rate':0.05,'num_leaves':31,'feature_fraction':0.8,'bagging_fraction':0.8,'bagging_freq':1,'seed':42}
final_lgb = lgb.train(pall, dall, num_boost_round=200)  # fixed rounds ~ best from cv

pred_tr = (final_ridge.predict(Ttr) + final_lgb.predict(Xtr_num))/2
pred_tr = np.clip(pred_tr, 0, 5)
train_rmse = mean_squared_error(y, pred_tr)**0.5
train_pear, _ = pearsonr(y, pred_tr)
print(f'TRAIN RMSE (full fit): {train_rmse:.4f}')
print(f'TRAIN Pearson: {train_pear:.4f}')

In [ ]:
# test predictions
base_te = (final_ridge.predict(Tte) + final_lgb.predict(Xte_num))/2
# v2 weighted ensemble if available
try:
    parts = {}
    parts['ridge'] = final_ridge.predict(Tte)
    parts['lgb'] = final_lgb.predict(Xte_num)
    parts['sbert'] = pred_sbert_te
    if 'pred_grammar_te' in globals():
        parts['grammar'] = pred_grammar_te
    if 'pred_bert_te' in globals():
        parts['bert'] = pred_bert_te
    w = globals().get('ENSEMBLE_WEIGHTS', None)
    if w is not None:
        order = list(w.keys())
        pred_te = sum(parts[k]*w[k] for k in order)
        print('used ensemble weights', w)
    else:
        pred_te = base_te
except Exception as e:
    print('ensemble fallback', e)
    pred_te = base_te
pred_te = np.clip(pred_te, 0, 5)
# v2: rubric is 0.5 steps, rounding helps RMSE
pred_te = np.round(pred_te*2)/2
sub = pd.DataFrame({'filename': order_te, 'label': pred_te})
print(sub.head(10))
print(sub.describe())
sub.to_csv('submission.csv', index=False)
print('saved submission.csv', sub.shape)
plt.hist(pred_te, bins=20, edgecolor='black')
plt.title('Test prediction distribution')
plt.xlabel('pred score')
plt.show()

## 6. Short report
**Problem:** predict Likert grammar score 0-5 from 45-60s speech.

**Preprocessing:** load at 16kHz mono, trim silence (top_db=30), peak-normalize to 0.9. No heavy denoising - whisper is already robust.

**Why transcription first:** rubric is about sentence structure / syntax / grammatical accuracy, which is textual. Acoustic-only models can't see grammar. So whisper-base -> text, then text regression. Acoustic features (MFCC, ZCR, energy, silence ratio) added as fallback for fluency/pauses.

**Features:** (a) TF-IDF 1-2grams 5k on transcripts, (b) 11 linguistic stats (word count, WPM, TTR, words/sent, stopword ratio...), (c) 30 acoustic stats. Ridge handles sparse TF-IDF, LightGBM handles dense numerics, average = ensemble.

**Eval:** 5-fold CV RMSE + Pearson (kaggle metric). Full-train RMSE printed above (required). Plots: label hist, duration-vs-score, pred-vs-true, residuals, importances.

**Limits / next:** whisper errors on heavy accents; rubric 0 vs 1 edge cases are rare (37 zeros); could try fine-tuning BERT on transcripts or wav2vec2 embeddings if time permits. Kept this version because I can explain every step.

**Outputs:** `submission.csv` (filename,label clipped 0-5) + `transcripts_*.csv` cache.

In [ ]:
print('done. files:', os.listdir('.')[:20])